In [1]:
import requests
import time

In [2]:
# retrieve publication data from semantic scholar database using api
# api_key might expire/break; make sure api_key is working properly

def get_publication_data(api_key, query, target, limit, offset):
    endpoint = 'https://api.semanticscholar.org/graph/v1/paper/search'
    headers = {'x-api-key': api_key}
    params = {
        'query': query,
        'limit': limit,
        'offset': offset,

        'fields': ','.join([
            'paperId',
            'externalIds',  # this dictionary object contains 'DOI' key
            'title',
            'citationCount',
            'influentialCitationCount',
            'isOpenAccess',
            'openAccessPdf'  # this dictionary object contains 'url' key
        ])
    }

    delay = 5
    last_offset = 0
    next_offset = 0
    open_access_publications = []
    
    while len(open_access_publications) < target:
        response = requests.get(
            endpoint,
            params=params,
            headers=headers
        )
    
        if response.status_code == 200:
            data = response.json()
            publications = data['data']
            last_offset = data['offset']
            next_offset = data['next']
            
            for pub in publications:
                if pub['isOpenAccess']:
                    open_access_publications.append(pub)
            
            params['offset'] += params['limit']
            print(f'Collected data: {len(open_access_publications)}')
        
        elif response.status_code == 429:
            print(f'Too many requests! Waiting for {delay}s...')
            time.sleep(delay)
        
        else:
            print(f'Error: {response.status_code}! Waiting for {delay}s...')
            time.sleep(delay)

    if len(open_access_publications) >= target:
        print('\nTarget reached successfully.')
        print(f'Total collected data: {len(open_access_publications)}')
        print(f'Last offset used: {last_offset}')
        print(f'Next offset to set: {next_offset}')

    return open_access_publications[:target]


In [61]:
# download pdf from its source 
# downloaded pdfs may still corrupted (unable to open)

def get_pdf(publication_list):
    success_list = []
    failed_list = []
    null_url_list = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for pub in publication_list:
        if pub['url'] != '':
            url = pub['url']
            
            try:
                response = requests.get(url, headers=headers, timeout=20)
    
                if response.status_code == 200:  # status code 200 is successful response
                    pdf_content = response.content
    
                    with open(f'pdfs/{pub['paper_id']}.pdf', 'wb') as pdf_file:
                        pdf_file.write(pdf_content)
    
                    success_list.append(pub)
    
                else:
                    failed_list.append(pub)
                    print(f'{response.status_code}:{response.reason}')

            except requests.exceptions.RequestException as e:
                failed_list.append(pub)
                print(f'Request failed: {e}')

            time.sleep(2)
            
        else:
            print(f'No URL found!')
            null_url_list.append(pub)

    print(f'\nSuccessful download: {len(success_list)}')
    print(f'Failed download: {len(failed_list)}')
    print(f'No URL found: {len(null_url_list)}')
    
    return success_list, failed_list, null_url_list
    

In [4]:
# set the api_key

def main():
    search_keys = [
        'cellulosic materials',
        'cellulose based materials',
        'nanocellulose',
        'nanocrystalline cellulose',
        'cellulose nanocrystals',
        'cellulose nanomaterials',
        'cellulose nanofibers',
        'cellulose nanofibrils',
        'nanofibrillated cellulose',
        'microfibrillated cellulose',
        'bacterial cellulose',
        'regenerated cellulose',
        'cellulose composites',
        'cellulose fibers',
        'cellulosic membranes',
        'cellulose films',
        'cellulose aerogels',
        'cellulose hydrogels',
        'cellophane',
        'rayon',
    ]
    api_key = 's2k-oet8aLS8w1jdlRWNaRzeWlkhqM8HY1ilMbopmrVf'
    target = 100  # number of open access publication data to retrieve
    limit = 25    # number of publication data retrieves per api request
    offset = 0    # starting index to retrieve publication data
    
    global all_publications
    all_publications = []
    
    for search_key in search_keys:
        print(f'\nSending request for search key =>> {search_key}')
        print('==================================================')
        publication_list = get_publication_data(
            api_key=api_key,
            query=search_key,
            target=target,
            limit=limit,
            offset=offset
        )

        print('Processing retrived data...')
        for paper in publication_list:
            pdf_info = paper.get('openAccessPdf') or {}
            all_publications.append({
                'search_key': search_key,
                'paper_id': paper['paperId'],
                'doi': paper['externalIds']['DOI'],
                'title': paper['title'],
                'citation_count': paper['citationCount'],
                'influential_citation_count': paper['influentialCitationCount'],
                'is_open_access': paper['isOpenAccess'],
                'url': pdf_info.get('url') or ''
            })
        print('Processed and stored successfully!')

    print('\nFinding unique papers and counting their occurrences...')
    unique_papers = {}

    for paper in all_publications:
        paper_id = paper['paper_id']
    
        if paper_id in unique_papers:
            unique_papers[paper_id]['occurrence'] += 1
        else:
            unique_papers[paper_id] = paper.copy()
            unique_papers[paper_id]['occurrence'] = 1
    
    all_publications = list(unique_papers.values())
    print('Unique papers identified and occurrences counted successfully!')


if __name__ == '__main__':
    main()


Sending request for search key =>> cellulosic materials
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Collected data: 11
Too many requests! Waiting for 5s...
Collected data: 22
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Collected data: 30
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Collected data: 37
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Collected data: 41
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Too many requests! Waiting for 5s...
Collected data: 43
Too many reques

In [79]:
print(len(all_publications))

1786


In [71]:
# sort publication by their occurrence

list_by_occurrence = all_publications.copy()
list_by_occurrence.sort(
    key=lambda paper: paper['occurrence'],
    reverse=True
)
list_by_occurrence_f100 = list_by_occurrence[:100]

In [72]:
# sort publication by their influentialCitationCount

list_by_influential = all_publications.copy()
list_by_influential.sort(
    key=lambda paper: paper['influential_citation_count'],
    reverse=True
)
list_by_influential_f100 = list_by_influential[:100]

In [73]:
# sort publication by their citationCount

list_by_citation = all_publications.copy()
list_by_citation.sort(
    key=lambda paper: paper['citation_count'],
    reverse=True
)
list_by_citation_f100 = list_by_citation[:100]

In [74]:
# combine the three top-100 lists, keeping only the first occurrence of each paper_id

final_list = []
seen_ids = set()

for paper in (list_by_occurrence_f100 + list_by_influential_f100 + list_by_citation_f100):
    if paper['paper_id'] not in seen_ids:
        final_list.append(paper)
        seen_ids.add(paper['paper_id'])

print('Total unique papers:', len(final_list))

Total unique papers: 226


In [80]:
final_list

[{'search_key': 'cellulosic materials',
  'paper_id': 'e7c08c9f62aa82c6d9b3d37010c47b1e30a55bf7',
  'doi': '10.1039/c7ra11157f',
  'title': 'Current progress in production of biopolymeric materials based on cellulose, cellulose nanofibers, and cellulose derivatives',
  'citation_count': 382,
  'influential_citation_count': 5,
  'is_open_access': True,
  'url': 'https://doi.org/10.1039/c7ra11157f',
  'occurrence': 5},
 {'search_key': 'cellulosic materials',
  'paper_id': '1f90ab1392d01dfe6019c477198b7da2ceb6801c',
  'doi': '10.1007/s10570-023-05172-1',
  'title': 'Cellulose materials with high light transmittance and high haze: a review',
  'citation_count': 40,
  'influential_citation_count': 0,
  'is_open_access': True,
  'url': 'https://www.researchsquare.com/article/rs-1310113/latest.pdf',
  'occurrence': 4},
 {'search_key': 'cellulose based materials',
  'paper_id': '33b7bd9dd36313d561792b7a592fcb219365d121',
  'doi': '10.1007/s10924-019-01621-w',
  'title': 'Enzymatic Degradation 

In [75]:
# minimum and maximum values for occurrence, citation count, and influential citation count

for key in ['occurrence', 'citation_count', 'influential_citation_count']:
    values = [paper[key] for paper in final_list]
    print(f'{key}: min = {min(values)}, max = {max(values)}')

occurrence: min = 1, max = 5
citation_count: min = 25, max = 2755
influential_citation_count: min = 0, max = 72


In [70]:
# split final_list into two batches: citationCount >= 330 in batch 1 and citationCount < 330 in batch 2

final_list_batch1 = []
final_list_batch2 = []

for paper in final_list:
    if paper['citation_count'] >= 330:
        final_list_batch1.append(paper)
    else:
        final_list_batch2.append(paper)

print('Batch 1:', len(final_list_batch1))
print('Batch 2:', len(final_list_batch2))

Batch 1: 100
Batch 2: 126


In [77]:
final_list_batch1

[{'search_key': 'cellulosic materials',
  'paper_id': 'e7c08c9f62aa82c6d9b3d37010c47b1e30a55bf7',
  'doi': '10.1039/c7ra11157f',
  'title': 'Current progress in production of biopolymeric materials based on cellulose, cellulose nanofibers, and cellulose derivatives',
  'citation_count': 382,
  'influential_citation_count': 5,
  'is_open_access': True,
  'url': 'https://doi.org/10.1039/c7ra11157f',
  'occurrence': 5},
 {'search_key': 'microfibrillated cellulose',
  'paper_id': '939a10e3d8ade88e9699cb2a04143e18f5a3479e',
  'doi': '10.1007/S10570-010-9405-Y',
  'title': 'Microfibrillated cellulose and new nanocomposite materials: a review',
  'citation_count': 2755,
  'influential_citation_count': 60,
  'is_open_access': True,
  'url': 'http://orbit.dtu.dk/en/publications/microfibrillated-cellulose-and-new-nanocomposite-materials-a-review(b016fe2d-6365-4cd4-88e9-ee99016882ff).html',
  'occurrence': 3},
 {'search_key': 'cellulosic materials',
  'paper_id': '5e15357bbde7c10a7706498d772ba096

In [78]:
final_list_batch2

[{'search_key': 'cellulosic materials',
  'paper_id': '1f90ab1392d01dfe6019c477198b7da2ceb6801c',
  'doi': '10.1007/s10570-023-05172-1',
  'title': 'Cellulose materials with high light transmittance and high haze: a review',
  'citation_count': 40,
  'influential_citation_count': 0,
  'is_open_access': True,
  'url': 'https://www.researchsquare.com/article/rs-1310113/latest.pdf',
  'occurrence': 4},
 {'search_key': 'cellulose based materials',
  'paper_id': '33b7bd9dd36313d561792b7a592fcb219365d121',
  'doi': '10.1007/s10924-019-01621-w',
  'title': 'Enzymatic Degradation and Pilot-Scale Composting of Cellulose-Based Films with Different Chemical Structures',
  'citation_count': 113,
  'influential_citation_count': 2,
  'is_open_access': True,
  'url': 'https://link.springer.com/content/pdf/10.1007/s10924-019-01621-w.pdf',
  'occurrence': 4},
 {'search_key': 'cellulose based materials',
  'paper_id': '579caa2d6245eb87276e8d4ad89449e7cd1839ee',
  'doi': '10.3390/nano15050356',
  'title'

In [82]:
# save the three lists as separate JSON files

import json

for name, data in [
    ('final_list_batch1', final_list_batch1),
    ('final_list_batch2', final_list_batch2),
    ('final_list', final_list)
]:
    with open(f'{name}.json', 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)